<!-- NB_DOC_INTRO_V1 -->
# Detection d'outliers — methodes statistiques + ML

> 📚 **Doc thematique** : [docs/02_EDA.md](docs/02_EDA.md) (EDA & Visualisation)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Les **outliers** (valeurs aberrantes) peuvent **gravement biaiser** un modele lineaire, fausser des moyennes, masquer des patterns. Mais **les supprimer aveuglement** peut faire perdre du signal precieux (fraud, anomalie systeme, evenement rare).

Ce notebook execute **7 methodes** de detection (IQR, Z-score, Elliptic Envelope, One-Class SVM, LOF, Isolation Forest, DBSCAN) sur un dataset synthetique 2D + benchmark.

## Plan

1. Setup + dataset jouet avec outliers connus
2. Methodes statistiques univariees (IQR, Z-score)
3. Elliptic Envelope (gaussien multivarie)
4. One-Class SVM (frontiere non-lineaire)
5. Local Outlier Factor (densite locale)
6. Isolation Forest (ensemble)
7. DBSCAN-based
8. Comparatif final + ROC
9. Pieges et anti-patterns
10. References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.covariance import EllipticEnvelope
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# === Dataset : 2 clusters gaussiens + 20 outliers uniformes ===
X_inliers, _ = make_blobs(n_samples=400, centers=[(-2, -2), (2, 2)],
                          cluster_std=0.5, random_state=SEED)
X_outliers = np.random.uniform(low=-6, high=6, size=(20, 2))
X = np.vstack([X_inliers, X_outliers])
y_true = np.array([1]*400 + [-1]*20)   # 1 = inlier, -1 = outlier (convention sklearn)
contamination = 20 / len(X)

print(f"N points : {len(X)}, dont outliers : {(y_true == -1).sum()}")
print(f"Contamination : {contamination:.3f}")

plt.figure(figsize=(7, 6))
plt.scatter(X[y_true == 1, 0], X[y_true == 1, 1], c="C0", label="inlier", s=30, alpha=0.5)
plt.scatter(X[y_true == -1, 0], X[y_true == -1, 1], c="C3", label="outlier (vrai)", s=80, marker="x")
plt.title("Dataset")
plt.legend()
plt.show()

## 1. IQR univarie

Methode classique : `[Q1 - 1.5×IQR, Q3 + 1.5×IQR]` comme limites. Robuste, ne suppose pas la normalite.

In [ ]:
def iqr_outliers(x):
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    return (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)

# Sur chaque dimension separement, OR
out_iqr_dim0 = iqr_outliers(X[:, 0])
out_iqr_dim1 = iqr_outliers(X[:, 1])
mask_iqr = out_iqr_dim0 | out_iqr_dim1

print(f"Outliers IQR : {mask_iqr.sum()} (vrai : {(y_true == -1).sum()})")
print(f"  Vrais positifs : {((mask_iqr) & (y_true == -1)).sum()}")
print(f"  Faux positifs  : {((mask_iqr) & (y_true == 1)).sum()}")

## 2. Z-score

`|x - mean| / std > k` (typiquement k=3). Suppose distribution proche normale.

In [ ]:
from scipy import stats

z = np.abs(stats.zscore(X, axis=0))
mask_z = (z > 3).any(axis=1)
print(f"Outliers Z-score (k=3) : {mask_z.sum()}")
print(f"  Vrais positifs : {((mask_z) & (y_true == -1)).sum()}")

## 3. Elliptic Envelope — gaussien multivarie

Estime moyenne + covariance **robuste** (Minimum Covariance Determinant), trace une ellipse de confiance.

In [ ]:
ee = EllipticEnvelope(contamination=contamination, random_state=SEED).fit(X)
pred_ee = ee.predict(X)
print(f"Outliers EllipticEnvelope : {(pred_ee == -1).sum()}")
print(f"  Accuracy vs vrai : {(pred_ee == y_true).mean():.3f}")

## 4. One-Class SVM

Apprend une **frontiere non-lineaire** englobant les inliers. Sensible aux hyperparams (`nu`, `gamma`).

In [ ]:
ocsvm = OneClassSVM(nu=contamination, gamma="auto").fit(X)
pred_svm = ocsvm.predict(X)
print(f"Outliers OneClassSVM : {(pred_svm == -1).sum()}")
print(f"  Accuracy vs vrai : {(pred_svm == y_true).mean():.3f}")

## 5. Local Outlier Factor (LOF)

Mesure la **densite locale** : un point est outlier si sa densite est **plus faible** que celle de ses voisins.

In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination)
pred_lof = lof.fit_predict(X)
print(f"Outliers LOF : {(pred_lof == -1).sum()}")
print(f"  Accuracy vs vrai : {(pred_lof == y_true).mean():.3f}")

## 6. Isolation Forest

**Plus rapide** que LOF, **scale** sur gros volumes. Idee : un outlier est isole par moins de splits aleatoires.

In [ ]:
iforest = IsolationForest(contamination=contamination, random_state=SEED, n_estimators=100)
pred_if = iforest.fit_predict(X)
print(f"Outliers IsolationForest : {(pred_if == -1).sum()}")
print(f"  Accuracy vs vrai : {(pred_if == y_true).mean():.3f}")

## 7. DBSCAN-based

Clustering densite : les points qui ne sont assignes **a aucun cluster** sont consideres outliers.

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5).fit(X)
pred_db = np.where(dbscan.labels_ == -1, -1, 1)
print(f"Outliers DBSCAN : {(pred_db == -1).sum()}")
print(f"  Accuracy vs vrai : {(pred_db == y_true).mean():.3f}")

## 8. Comparatif visuel

In [ ]:
methods = [
    ("IQR univarie", np.where(mask_iqr, -1, 1)),
    ("Z-score", np.where(mask_z, -1, 1)),
    ("EllipticEnvelope", pred_ee),
    ("OneClassSVM", pred_svm),
    ("LOF", pred_lof),
    ("IsolationForest", pred_if),
    ("DBSCAN", pred_db),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.ravel()

# Ground truth
axes[0].scatter(X[y_true == 1, 0], X[y_true == 1, 1], c="C0", s=20, alpha=0.4, label="inlier")
axes[0].scatter(X[y_true == -1, 0], X[y_true == -1, 1], c="C3", s=80, marker="x", label="outlier")
axes[0].set_title("Ground truth")
axes[0].legend(loc="upper right", fontsize=8)

# Methodes
for ax, (name, pred) in zip(axes[1:], methods):
    tp = (pred == -1) & (y_true == -1)
    fp = (pred == -1) & (y_true == 1)
    tn = (pred == 1) & (y_true == 1)
    fn = (pred == 1) & (y_true == -1)
    ax.scatter(X[tn, 0], X[tn, 1], c="lightblue", s=20, alpha=0.5, label="TN")
    ax.scatter(X[tp, 0], X[tp, 1], c="C2", s=80, marker="x", label="TP (outlier ok)")
    ax.scatter(X[fp, 0], X[fp, 1], c="orange", s=40, marker="s", label="FP (inlier flag)")
    ax.scatter(X[fn, 0], X[fn, 1], c="C3", s=80, marker="d", label="FN (outlier rate)")
    ax.set_title(f"{name}\nacc={((pred == y_true).mean()):.2f}")
    ax.legend(loc="upper right", fontsize=7)

plt.tight_layout()
plt.show()

## 9. Benchmark — precision/recall

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Convention : outlier = classe positive (1)
y_true_pos = (y_true == -1).astype(int)
results = []
for name, pred in methods:
    pred_pos = (pred == -1).astype(int)
    results.append({
        "method": name,
        "n_flagged": pred_pos.sum(),
        "precision": precision_score(y_true_pos, pred_pos, zero_division=0),
        "recall":    recall_score(y_true_pos, pred_pos, zero_division=0),
        "f1":        f1_score(y_true_pos, pred_pos, zero_division=0),
    })
df_results = pd.DataFrame(results).sort_values("f1", ascending=False)
print(df_results.to_string(index=False))

## 10. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Supprimer les outliers aveuglement | Investiguer : data quality issue ou vrai signal (fraud) ? |
| Z-score sur distrib log-normale | Utiliser IQR ou transformer en log d'abord |
| `contamination` non specifie ou faux | Estimer via domaine ou laisser sklearn `auto` |
| IsolationForest sans `random_state` | Reproductibilite cassee |
| OCSVM avec `gamma="scale"` par defaut sur 2D | Tester plusieurs gammas, ou utiliser `nu` |
| Confondre outliers (univarie) et **anomalies contextuelles** (TS) | TS : utiliser ADTK, Prophet, ou Autoencoder |
| Eval sans ground truth | C'est non-supervise → utiliser metriques internes (silhouette inverse) |
| Pas standardiser avant LOF / OCSVM | Variables a grandes echelles dominent |
| Detection en train, applique sur test sans persister | Sauvegarder le modele fit |
| 1 seule methode | Combiner (ensemble outlier detection) |

## 11. References

- **PyOD** : 30+ algos, recommande pour prod : https://pyod.readthedocs.io/
- **sklearn outlier detection** : https://scikit-learn.org/stable/modules/outlier_detection.html
- *Outlier Analysis* (Aggarwal, 2017)
- **Anomaly detection TS** : ADTK, Prophet anomaly, Twitter AnomalyDetection

### Voir aussi (collection)
- [Structures_Preprocessing.ipynb](Structures_Preprocessing.ipynb) — outliers en pipeline
- [EDA_Stats_Analyse_Desc_Visual.ipynb](EDA_Stats_Analyse_Desc_Visual.ipynb)
- [MLOps_Drift_Monitoring.ipynb](MLOps_Drift_Monitoring.ipynb)
